# 06 · Parques a revisar

¿Qué parque conviene mirar primero? Se compara cada parque con lo que debería haber generado y con sus vecinos de provincia. Una señal no es una falla confirmada.

In [ ]:
import json, sys
from pathlib import Path
sys.path.insert(0, str(Path.cwd().parent))
import numpy as np, pandas as pd, matplotlib.pyplot as plt
from src.config import DATA_PROCESSED
from src.senales import marcar, sensibilidad_umbral
D = json.loads((DATA_PROCESSED / 'defensa_evidencia.json').read_text())
S = D['senales']
d = pd.read_parquet(DATA_PROCESSED / 'defensa_argentina.parquet')
s = marcar(d)
pd.Series(S['resumen'])[['meses','marcados','sitios','gwh_faltante','gwh_generada','pct','usd_millones']]

## Lo esperado sale de un modelo que no vio ese parque

Cinco grupos de parques. Para cada grupo se entrena hasta 2023 con los demás. Así una caída del parque no contamina su propia referencia.

In [ ]:
t = s[s['fecha'].ge('2025-01-01')]
fig, ax = plt.subplots(figsize=(7, 5))
ok = t[~t['senal']]; m = t[t['senal']]
ax.scatter(ok['rinde_modelo'], ok['rinde_vecinos'], s=8, alpha=.3, c='gray', label='sin señal')
ax.scatter(m['rinde_modelo'], m['rinde_vecinos'], s=14, c='k', label='señal')
ax.axvline(.8, c='b', ls='--'); ax.axhline(.8, c='b', ls='--')
ax.set(xlabel='Rinde contra el modelo', ylabel='Rinde contra los vecinos', xlim=(0, 2), ylim=(0, 2)); ax.legend();

## El ranking

In [ ]:
pd.DataFrame(S['ranking'])[['nombre','provincia','desde','hasta','meses','rinde_modelo','rinde_vecinos','gwh_faltante']].round(2)

In [ ]:
c = S['ranking'][0]['central']
g = s[s['central'].eq(c)].sort_values('fecha'); f = pd.to_datetime(g['fecha'])
fig, ax = plt.subplots(figsize=(10, 3.5))
ax.plot(f, g['factor_capacidad'], c='k', label='observado'); ax.plot(f, g['pred_oof'], c='b', ls=':', label='esperado')
ax.scatter(f[g['senal']], g.loc[g['senal'], 'factor_capacidad'], c='k', marker='s', label='señal')
ax.set_title(g['central_descripcion'].iloc[0]); ax.legend();

## Qué tan sensible es la regla

Cambiar el umbral mueve poco la lista. Las caídas simuladas solo miden la regla: no entrenan nada.

In [ ]:
display(pd.DataFrame(S['sensibilidad']).round(1)); display(pd.DataFrame(S['caidas']))

## Dos meses, una zona: cuando bajan todos

Si caen todos los parques de una provincia, el problema no es de un parque. Ejemplo: Cauchari, septiembre y octubre de 2020.

In [ ]:
x = s[s['central_descripcion'].str.contains('CAUCHARI') & s['fecha'].isin(['2020-09-01', '2020-10-01'])]
x[['central_descripcion','fecha','rinde_modelo','rinde_vecinos','n_vecinos','estado']].round(2)

## K-Means, PCA y DBSCAN

**K-Means** agrupa parques por su perfil anual; la silueta elige k. **PCA** resume los doce meses en dos ejes para poder dibujarlos. **DBSCAN** busca meses raros por densidad, sin mirar la regla.

In [ ]:
T = S['tipos_parque']
display(pd.Series(T['silueta'], name='silueta').round(3)); display(pd.DataFrame(T['grupos']).round(3))
p = pd.DataFrame(T['puntos'])
plt.figure(figsize=(6, 4)); plt.scatter(p['x'], p['y'], c=p['grupo'], cmap='coolwarm')
plt.xlabel(f"PC1 ({T['pca_varianza'][0]:.0%})"); plt.ylabel(f"PC2 ({T['pca_varianza'][1]:.0%})");

In [ ]:
R = S['meses_raros']; R

**Lectura.** DBSCAN deja afuera de su grupo principal solo una parte de las señales, y la mayoría de lo que aparta son meses que rinden *más*. Los meses con señal forman una cola continua, no islas: la densidad no los separa. Por eso se usa la regla explícita: es más simple de explicar y mide lo que el negocio pregunta. K-Means sirve para describir, no para decidir: separa parques por cuánto rinden en el año.